# Pipeline de Arquitectura Medallion

## Objetivos
Este notebook orquesta la arquitectura medallion completa para datos de inversiones del MEF:
1. **Capa Bronze**: Cargar datasets crudos tal como vienen desde `data/raw/`
2. **Capa Silver**: Limpiar, normalizar y deduplicar usando reglas de negocio
3. **Capa Gold**: Construir modelo dimensional (esquema estrella) y vistas para Tableau
4. **Bitácora**: Documentar todas las transformaciones por capa

## Flujo de Datos
- **Entrada**: 5 archivos CSV crudos del portal MEF (detalle, F12B, estado, proceso, cierre)
- **Salida**: Reportes de profiling, datasets limpios, modelo dimensional, vistas para Tableau
- **Framework**: Polars (optimizado para datasets grandes de hasta 5.6M filas)
- **Grano clave**: CODIGO_UNICO (CUI) + PERIODO para análisis temporal

## Capas Medallion

### Bronze (Crudo)
- Sin transformaciones
- Ubicación: `data/raw/{dataset_slug}/data.csv`

### Silver (Limpio)
- Estandarizar CODIGO_UNICO (remover decimales)
- Normalizar PERIODO (YYYYMM → YYYY-MM)
- Convertir columnas numéricas (MONTO, DEVENGADO, etc.)
- Parsear columnas de fecha (FECHA_*, FEC_*)
- Eliminar duplicados por claves
- Ubicación: `data/processed/silver/{dataset_slug}/data.csv`
- Bitácora: `outputs/bitacora/bitacora_bronze_silver.csv`

### Gold (Analítico)
- Construcción del modelo dimensional:
  - **dim_inversion**: Metadatos de la inversión (CUI, nombre, sector, ubicación)
  - **dim_cierre**: Información de cierre (cuándo, motivo, monto)
  - **fact_financiera**: Ejecución financiera (F12B, por CUI+PERIODO)
  - **fact_situacional**: Actualizaciones situacionales (por CUI+PERIODO)
  - **fact_componentes**: Ejecución por componentes (por CUI+componente+PERIODO)
  - **tableau_inversion_period_full**: Vista agregada completa (con flags de calidad)
  - **tableau_inversion_period_clean**: Vista filtrada para análisis (solo registros válidos)
- Ubicación: `data/processed/gold/`
- Bitácora: `outputs/bitacora/bitacora_silver_gold.csv`

## Reglas de Negocio Clave
- **Identificación de inversión**: Todos los registros deben tener CODIGO_UNICO
- **Granularidad temporal**: Mensual (YYYYMM → YYYY-MM)
- **Datos financieros**: Agregados por CUI+PERIODO desde F12B
- **Deduplicación**: Se mantiene la primera ocurrencia cuando hay duplicados

In [1]:
import sys
from pathlib import Path
import polars as pl

if 'medallion' in sys.modules:
    del sys.modules['medallion']
for key in list(sys.modules.keys()):
    if key.startswith('medallion.'):
        del sys.modules[key]

notebook_dir = Path.cwd()
if 'notebooks' in notebook_dir.parts:
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

sys.path.insert(0, str(project_root / 'scripts'))

from medallion.common import DATASET_CONFIGS, RAW_DIR, PROCESSED_DIR, OUTPUTS_DIR, ensure_dir
from medallion.profiling import profile_all_datasets
from medallion.bronze_to_silver import run_pipeline as run_bronze_to_silver
from medallion.silver_to_gold import run_pipeline as run_silver_to_gold

ensure_dir(OUTPUTS_DIR / 'profiling')
ensure_dir(OUTPUTS_DIR / 'bitacora')
ensure_dir(PROCESSED_DIR / 'silver')
ensure_dir(PROCESSED_DIR / 'gold')

print(f"✓ Setup complete")
print(f"  Project Root: {project_root}")
print(f"  Scripts: {project_root / 'scripts'}")
print(f"  Raw Data: {RAW_DIR}")
print(f"  Outputs: {OUTPUTS_DIR}")
print(f"  Datasets: {len(DATASET_CONFIGS)}")
for config in DATASET_CONFIGS:
    print(f"    - {config.slug}")

✓ Setup complete
  Project Root: c:\Users\jayka\Documents\Projects\data-visualization-tf
  Scripts: c:\Users\jayka\Documents\Projects\data-visualization-tf\scripts
  Raw Data: c:\Users\jayka\Documents\Projects\data-visualization-tf\data\raw
  Outputs: c:\Users\jayka\Documents\Projects\data-visualization-tf\outputs
  Datasets: 5
    - detalle-de-inversiones
    - formato-12b-de-de-inversiones
    - estado-situacional-de-inversiones
    - proceso-de-seleccion-de-inversiones
    - cierre-de-inversiones


## Capa Bronze: Perfilamiento

Analizar los datasets crudos para evaluar la calidad de los datos:
- Tipos de datos por columna
- Conteo y porcentaje de valores nulos
- Cardinalidad (valores únicos)
- Duplicados

**Salida**: Reportes de perfilamiento en `outputs/profiling/`

In [2]:
# Profile all datasets
dataset_slugs = [config.slug for config in DATASET_CONFIGS]
profiling_result = profile_all_datasets(dataset_slugs)

print(f"✓ Profiling complete for {len(profiling_result['profiles'])} datasets")
print(f"  Bitácora: {profiling_result['bitacora_path']}")

# Display sample profiles
print("\n=== Detalle de Inversiones Profile ===")
detalle_profile = profiling_result['profiles'].get('detalle-de-inversiones')
if detalle_profile:
    df = detalle_profile['profile_df']
    print(f"Rows: {df.shape[0]} columns")
    print(df.head(10))
    print(f"\nNull Summary:\n{df.select(['column', 'null_pct']).filter(pl.col('null_pct') > 0)}")

✓ Profiling complete for 5 datasets
  Bitácora: c:\Users\jayka\Documents\Projects\data-visualization-tf\outputs\bitacora\bitacora_profiling.csv

=== Detalle de Inversiones Profile ===
Rows: 68 columns
shape: (10, 8)
┌─────────────┬────────┬────────────┬────────────┬──────────┬────────────┬────────────┬────────────┐
│ column      ┆ dtype  ┆ non_null_c ┆ null_count ┆ null_pct ┆ unique_cou ┆ duplicate_ ┆ cardinalit │
│ ---         ┆ ---    ┆ ount       ┆ ---        ┆ ---      ┆ nt         ┆ count      ┆ y_pct      │
│ str         ┆ str    ┆ ---        ┆ i64        ┆ f64      ┆ ---        ┆ ---        ┆ ---        │
│             ┆        ┆ i64        ┆            ┆          ┆ i64        ┆ i64        ┆ f64        │
╞═════════════╪════════╪════════════╪════════════╪══════════╪════════════╪════════════╪════════════╡
│ NIVEL       ┆ String ┆ 260422     ┆ 0          ┆ 0.0      ┆ 3          ┆ 260419     ┆ 0.0        │
│ SECTOR      ┆ String ┆ 260422     ┆ 0          ┆ 0.0      ┆ 36         ┆ 26

## Capa Silver: Limpieza y Normalización

Aplicar reglas de negocio para transformar los datos crudos:
1. Eliminar registros con CODIGO_UNICO nulo  
2. Estandarizar CODIGO_UNICO (remover decimales)  
3. Normalizar PERIODO (YYYYMM → YYYY-MM)  
4. Convertir columnas numéricas  
5. Parsear columnas de fecha  
6. Eliminar duplicados según claves  

**Salida**: Datasets limpios en `data/processed/silver/`  
**Bitácora**: `outputs/bitacora/bitacora_bronze_silver.csv`

In [3]:
# Execute Bronze → Silver pipeline
silver_result = run_bronze_to_silver()

print(f"✓ Bronze → Silver transformation complete")
print(f"  Bitácora: {silver_result['bitacora']}")

# Display bitácora summary
if silver_result['bitacora'].exists():
    bitacora_df = pl.read_csv(silver_result['bitacora'])
    print(f"\n=== Transformation Summary ===")
    print(f"Total rules applied: {len(bitacora_df)}")
    print(f"\nAffected Rows by Rule:")
    summary = bitacora_df.group_by('rule').agg([
        pl.col('affected_rows').sum().alias('total_rows'),
        pl.col('dataset').n_unique().alias('datasets')
    ])
    print(summary)
    
    print(f"\n=== Bitácora Details ===")
    print(bitacora_df.select(['dataset', 'rule', 'affected_rows', 'note']))

c:\Users\jayka\Documents\Projects\data-visualization-tf\scripts\medallion\bronze_to_silver.py:26: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  if "CODIGO_UNICO" in scan.columns:
c:\Users\jayka\Documents\Projects\data-visualization-tf\scripts\medallion\bronze_to_silver.py:46: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  column for column, dtype in ldf.schema.items() if dtype == pl.Utf8
c:\Users\jayka\Documents\Projects\data-visualization-tf\scripts\medallion\bronze_to_silver.py:51: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names wit

✓ Bronze → Silver transformation complete
  Bitácora: c:\Users\jayka\Documents\Projects\data-visualization-tf\outputs\bitacora\bitacora_bronze_silver.csv

=== Transformation Summary ===
Total rules applied: 14

Affected Rows by Rule:
shape: (4, 3)
┌─────────────────────┬────────────┬──────────┐
│ rule                ┆ total_rows ┆ datasets │
│ ---                 ┆ ---        ┆ ---      │
│ str                 ┆ i64        ┆ u32      │
╞═════════════════════╪════════════╪══════════╡
│ parse_dates         ┆ 0          ┆ 5        │
│ coerce_numeric      ┆ 0          ┆ 5        │
│ drop_duplicate_keys ┆ 3513       ┆ 2        │
│ drop_null_cui       ┆ 3514       ┆ 2        │
└─────────────────────┴────────────┴──────────┘

=== Bitácora Details ===
shape: (14, 4)
┌──────────────────────────────┬─────────────────────┬───────────────┬─────────────────────────────┐
│ dataset                      ┆ rule                ┆ affected_rows ┆ note                        │
│ ---                        

## Gold Layer: Modelado Dimensional

Construcción de un esquema estrella para consultas analíticas y visualización en Tableau:

**Dimensiones**:
- **dim_inversion**: Metadatos estáticos de la inversión (Detalle)
- **dim_cierre**: Atributos de cierre (Cierre)

**Hechos**:
- **fact_financiera**: Ejecución financiera (F12B) por CUI+PERIODO  
- **fact_situacional**: Actualizaciones situacionales (Estado) por CUI+PERIODO  
- **fact_componentes**: Ejecución por componentes (Proceso) por CUI+componente+PERIODO  
- **tableau_inversion_period_full**: Vista preagregada completa con flags de calidad  
- **tableau_inversion_period_clean**: Vista filtrada para análisis (solo registros válidos)

**Salida**: Todos los CSV en `data/processed/gold/`  
**Bitácora**: `outputs/bitacora/bitacora_silver_gold.csv`

In [ ]:
# Execute Silver → Gold pipeline
gold_result = run_silver_to_gold()

print(f"✓ Silver → Gold transformation complete")
print(f"\nGold Layer Outputs:")
for key, path in gold_result.items():
    if key != 'bitacora':
        df = pl.read_csv(path)
        print(f"  {key}: {len(df)} rows, {len(df.columns)} columns")

# Display dimension samples
print(f"\n=== dim_inversion Sample ===")
dim_inv = pl.read_csv(gold_result['dim_inversion'])
print(f"Columns: {dim_inv.columns}")
print(dim_inv.head(3))

print(f"\n=== fact_financiera Sample ===")
fact_fin = pl.read_csv(gold_result['fact_financiera'])
print(f"Shape: {len(fact_fin)} rows, {len(fact_fin.columns)} columns")
print(f"Key columns: CODIGO_UNICO, PERIODO")
print(fact_fin.head(3).select(['CODIGO_UNICO', 'PERIODO', 'DEV_TOTAL', 'AVANCE_FISICO'] if 'DEV_TOTAL' in fact_fin.columns else fact_fin.columns[:5]))

print(f"\n=== tableau_inversion_period Sample ===")
tableau = pl.read_csv(gold_result['tableau_clean'])
print(f"Shape: {len(tableau)} rows for Tableau")
print(tableau.head(3))

✓ Silver → Gold transformation complete

Gold Layer Outputs:
  dim_inversion: 260354 rows, 14 columns
  dim_cierre: 105394 rows, 5 columns
  fact_financiera: 213214 rows, 54 columns
  fact_situacional: 1350449 rows, 6 columns
  fact_componentes: 5593594 rows, 14 columns
  tableau_full: 252761 rows, 22 columns
  tableau_clean: 139411 rows, 22 columns

=== dim_inversion Sample ===
Columns: ['CODIGO_UNICO', 'NOMBRE_INVERSION', 'ESTADO', 'SITUACION', 'SECTOR', 'FUNCION', 'PROGRAMA', 'SUBPROGRAMA', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'UBIGEO', 'LATITUD', 'LONGITUD']
shape: (3, 14)
┌──────────────┬─────────────────┬────────┬───────────┬───┬──────────┬────────┬─────────┬──────────┐
│ CODIGO_UNICO ┆ NOMBRE_INVERSIO ┆ ESTADO ┆ SITUACION ┆ … ┆ DISTRITO ┆ UBIGEO ┆ LATITUD ┆ LONGITUD │
│ ---          ┆ N               ┆ ---    ┆ ---       ┆   ┆ ---      ┆ ---    ┆ ---     ┆ ---      │
│ i64          ┆ ---             ┆ str    ┆ str       ┆   ┆ str      ┆ i64    ┆ f64     ┆ f64      │
│       

## Pipeline Summary

All transformations complete. Ready for analysis and Tableau visualization.

**Key Metrics**:
- Input datasets: 5 (detalle, F12B, estado, proceso, cierre)
- Bronze rows: ~7.3M total
- Silver output: Cleaned and deduplicated
- Gold output: 6 tables (2 dims + 4 facts)
- Tableau ready: tableau_inversion_period.csv

**Next Steps**:
1. Review profiling reports in outputs/profiling/
2. Validate bitácora transformations in outputs/bitacora/
3. Load Gold tables into Tableau
4. Analyze execution vs. progress gap by investment

In [5]:
# Final summary
import os

print("=" * 60)
print("MEDALLION PIPELINE SUMMARY")
print("=" * 60)

# Count output files
profiling_dir = OUTPUTS_DIR / 'profiling'
if profiling_dir.exists():
    profile_files = list(profiling_dir.glob('*_profile.csv'))
    print(f"\n✓ Bronze Profiling: {len(profile_files)} reports")
    for f in profile_files:
        print(f"    {f.name}")

bitacora_dir = OUTPUTS_DIR / 'bitacora'
if bitacora_dir.exists():
    bitacora_files = list(bitacora_dir.glob('*.csv'))
    print(f"\n✓ Bitácora Files: {len(bitacora_files)}")
    for f in bitacora_files:
        print(f"    {f.name}")

silver_dir = PROCESSED_DIR / 'silver'
if silver_dir.exists():
    silver_files = list(silver_dir.glob('*/data.csv'))
    print(f"\n✓ Silver Layer: {len(silver_files)} cleaned datasets")
    for f in silver_files:
        print(f"    {f.parent.name}")

gold_dir = PROCESSED_DIR / 'gold'
if gold_dir.exists():
    gold_files = list(gold_dir.glob('*.csv'))
    print(f"\n✓ Gold Layer: {len(gold_files)} dimensional tables")
    for f in gold_files:
        df = pl.read_csv(f)
        print(f"    {f.name}: {len(df):>7,} rows")

MEDALLION PIPELINE SUMMARY

✓ Bronze Profiling: 5 reports
    cierre-de-inversiones_profile.csv
    detalle-de-inversiones_profile.csv
    estado-situacional-de-inversiones_profile.csv
    formato-12b-de-de-inversiones_profile.csv
    proceso-de-seleccion-de-inversiones_profile.csv

✓ Bitácora Files: 3
    bitacora_bronze_silver.csv
    bitacora_profiling.csv
    bitacora_silver_gold.csv

✓ Silver Layer: 5 cleaned datasets
    cierre-de-inversiones
    detalle-de-inversiones
    estado-situacional-de-inversiones
    formato-12b-de-de-inversiones
    proceso-de-seleccion-de-inversiones

✓ Gold Layer: 7 dimensional tables
    dim_cierre.csv: 105,394 rows
    dim_inversion.csv: 260,354 rows
    fact_componentes.csv: 5,593,594 rows
    fact_financiera.csv: 213,214 rows
    fact_situacional.csv: 1,350,449 rows
    tableau_inversion_period_clean.csv: 139,411 rows
    tableau_inversion_period_full.csv: 252,761 rows
